In [23]:
import os
import time
from pathlib import Path
from typing import List
import requests
import pandas as pd
from dotenv import load_dotenv


def load_azure_creds():
    """Load Azure Translator creds from .env, then fallback to azure key.txt."""
    load_dotenv()
    key = os.getenv("AZURE_TRANSLATOR_KEY")
    region = os.getenv("AZURE_TRANSLATOR_REGION")
    endpoint = os.getenv(
        "AZURE_TRANSLATOR_ENDPOINT",
        "https://api.cognitive.microsofttranslator.com",
    )

    key_file = Path("../../azure key.txt")
    if key_file.exists():
        for raw_line in key_file.read_text().splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("API_KEY") and not key:
                value = line.split("=", 1)[1].strip()
                key = value.strip('"').strip("'")
            if line.startswith("API_REGION") and not region:
                value = line.split("=", 1)[1].strip()
                region = value.strip('"').strip("'")

    if not key or not region:
        raise ValueError("Set AZURE_TRANSLATOR_KEY and AZURE_TRANSLATOR_REGION in .env or azure key.txt")
    return key, region, endpoint


AZURE_TRANSLATOR_KEY, AZURE_TRANSLATOR_REGION, AZURE_TRANSLATOR_ENDPOINT = load_azure_creds()

In [24]:
def translate_in_chunks(texts: List[str], chunk_size: int = 20) -> List[str]:
    results: List[str] = []
    for start in range(0, len(texts), chunk_size):
        chunk = texts[start : start + chunk_size]
        attempt = 0
        while True:
            try:
                translated = azure_translate(chunk, AZURE_TRANSLATOR_KEY, AZURE_TRANSLATOR_REGION, AZURE_TRANSLATOR_ENDPOINT)
                break
            except RuntimeError as err:
                err_text = str(err)
                if "429" in err_text or "request limits" in err_text:
                    backoff = min(60, 5 * (attempt + 1))
                    print(f"Hit rate limit; sleeping {backoff}s then retrying (attempt {attempt + 1})...")
                    time.sleep(backoff)
                    attempt += 1
                    if attempt >= 5:
                        raise
                else:
                    raise
        results.extend(translated)
    return results

In [25]:
df_xnli = pd.read_csv("../../datasets/cleaned/cleaned_xnli.csv")
print(f"Loaded XNLI rows: {len(df_xnli)}")

Loaded XNLI rows: 600


In [26]:
sentence1_list = df_xnli["sentence1"].to_list()
sentence2_list = df_xnli["sentence2"].to_list()

print("Translating sentence1 ...")
sentence1_translated = translate_in_chunks(sentence1_list)
print("Translating sentence2 ...")
sentence2_translated = translate_in_chunks(sentence2_list)

Translating sentence1 ...
Hit rate limit; sleeping 5s then retrying (attempt 1)...
Hit rate limit; sleeping 10s then retrying (attempt 2)...
Hit rate limit; sleeping 15s then retrying (attempt 3)...
Hit rate limit; sleeping 20s then retrying (attempt 4)...
Translating sentence2 ...
Hit rate limit; sleeping 5s then retrying (attempt 1)...
Hit rate limit; sleeping 10s then retrying (attempt 2)...
Hit rate limit; sleeping 15s then retrying (attempt 3)...
Hit rate limit; sleeping 20s then retrying (attempt 4)...


In [27]:
df_xnli["sentence1"] = sentence1_translated
df_xnli["sentence2"] = sentence2_translated

output_path = "../../datasets/translated/bing/bing_translated_xnli.csv"
df_xnli.to_csv(output_path, index=False)
print(f"Saved translated XNLI to {output_path}")

Saved translated XNLI to ../../datasets/translated/bing/bing_translated_xnli.csv
